# MotoGP Finishing Positions - Advanced Modeling and Analysis

This notebook performs advanced modeling and analysis on the finishing positions dataset following CRISP-DM methodology.

**Dataset Focus**: `riders_finishing_positions.csv`  
**CRISP-DM Phase**: 4 - Modeling  
**Input**: Prepared finishing positions data  
**Output**: Advanced statistical models and position-based insights

## Business Questions Addressed
- Q18: Pilotos com maior número de 4º, 5º, 6º que nunca entraram no pódio?
- Q19: Pilotos com maior número de 2º e 3º que nunca venceram?
- Advanced finishing position analysis
- Performance tier modeling

## 0. Setup and Data Loading

In [ ]:
# Import necessary libraries
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Configure plot styles
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

In [ ]:
# Load prepared finishing positions data
data_path = Path("../../data/interim/")
df = pd.read_csv(data_path / "finishing_positions_prepared.csv")

print(f"Prepared finishing positions data loaded: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nTotal riders: {len(df)}")
print(f"Riders with victories: {(df['victories'] > 0).sum()}")
print(f"Riders with podiums: {(df['total_podiums'] > 0).sum()}")
print(f"Riders with no podiums: {(df['total_podiums'] == 0).sum()}")
df.head()

## 1. Q18: Pilotos com maior número de 4º, 5º, 6º que nunca entraram no pódio?

Comprehensive analysis of riders who achieved consistent points finishes but never reached the podium.

In [ ]:
print("=== Q18: PILOTOS COM MAIS 4º-6º SEM PÓDIOS ===")

# Filter riders who never podiumed
no_podium_riders = df[df['total_podiums'] == 0].copy()

if len(no_podium_riders) > 0:
    print(f"📊 STATISTICAL OVERVIEW:")
    print(f"Total riders analyzed: {len(df)}")
    print(f"Riders who never reached podium: {len(no_podium_riders)} ({len(no_podium_riders)/len(df)*100:.1f}%)")
    print(f"Total 4th-6th finishes (no podium group): {no_podium_riders['non_podium_points'].sum():,}")
    print(f"Average 4th-6th per rider: {no_podium_riders['non_podium_points'].mean():.1f}")
    
    # Sort by non-podium points finishes
    no_podium_sorted = no_podium_riders.sort_values('non_podium_points', ascending=False)
    
    print(f"\nTop 20 pilotos com mais posições 4º-6º sem nunca subir ao pódio:")
    top_20 = no_podium_sorted.head(20)
    for i, (_, rider) in enumerate(top_20.iterrows(), 1):
        name = rider['rider_clean']
        country = rider['country_clean']
        fourth = int(rider['fourth_places'])
        fifth = int(rider['fifth_places'])
        sixth = int(rider['sixth_places'])
        total_points = int(rider['non_podium_points'])
        total_races = int(rider['total_races'])
        points_rate = (total_points / total_races) * 100 if total_races > 0 else 0
        print(f"{i:2d}. {name} ({country}): {total_points} posições (4º:{fourth}, 5º:{fifth}, 6º:{sixth}) em {total_races} corridas ({points_rate:.1f}%)")
    
    # Champion analysis
    champion = no_podium_sorted.iloc[0]
    champion_name = champion['rider_clean']
    champion_country = champion['country_clean']
    champion_points = int(champion['non_podium_points'])
    champion_fourth = int(champion['fourth_places'])
    champion_fifth = int(champion['fifth_places'])
    champion_sixth = int(champion['sixth_places'])
    champion_total_races = int(champion['total_races'])
    
    print(f"\n🏆 RESPOSTA Q18:")
    print(f"Piloto com mais posições 4º-6º sem pódios: {champion_name} ({champion_country})")
    print(f"Total de posições 4º-6º: {champion_points}")
    print(f"Detalhamento: 4º lugar = {champion_fourth}, 5º lugar = {champion_fifth}, 6º lugar = {champion_sixth}")
    print(f"Total de corridas: {champion_total_races}")
    
    # Advanced analysis of champion
    points_rate = (champion_points / champion_total_races) * 100
    consistency = champion_points / champion_total_races  # Points finish rate
    print(f"\nAnálise detalhada do {champion_name}:")
    print(f"  Taxa de pontuação: {points_rate:.1f}%")
    print(f"  Consistência: {consistency:.3f} (posições pontuáveis por corrida)")
    print(f"  Melhor posição alcançada: 4º lugar")
    
    # Position distribution for champion
    champion_position_dist = {
        '4º': champion_fourth,
        '5º': champion_fifth, 
        '6º': champion_sixth
    }
    
    favorite_position = max(champion_position_dist, key=champion_position_dist.get)
    print(f"  Posição mais frequente: {favorite_position} ({champion_position_dist[favorite_position]} vezes)")
    
    # Statistical analysis of the no-podium group
    print(f"\n📈 STATISTICAL ANALYSIS - NO PODIUM GROUP:")
    
    # Distribution stats
    print(f"Distribution of 4th-6th finishes:")
    print(f"  Mean: {no_podium_riders['non_podium_points'].mean():.1f}")
    print(f"  Median: {no_podium_riders['non_podium_points'].median():.1f}")
    print(f"  Std Dev: {no_podium_riders['non_podium_points'].std():.1f}")
    print(f"  Max: {no_podium_riders['non_podium_points'].max()}")
    
    # Percentile analysis
    percentiles = [50, 75, 90, 95, 99]
    print(f"\nPercentiles (4th-6th finishes):")
    for p in percentiles:
        value = np.percentile(no_podium_riders['non_podium_points'], p)
        print(f"  {p}th percentile: {value:.0f} finishes")
    
    # Position preference analysis
    print(f"\n🎯 POSITION PREFERENCE ANALYSIS:")
    total_fourth = no_podium_riders['fourth_places'].sum()
    total_fifth = no_podium_riders['fifth_places'].sum()
    total_sixth = no_podium_riders['sixth_places'].sum()
    total_positions = total_fourth + total_fifth + total_sixth
    
    print(f"Overall position distribution (no-podium group):")
    print(f"  4th place: {total_fourth:,} ({total_fourth/total_positions*100:.1f}%)")
    print(f"  5th place: {total_fifth:,} ({total_fifth/total_positions*100:.1f}%)")
    print(f"  6th place: {total_sixth:,} ({total_sixth/total_positions*100:.1f}%)")
    
    # Identify 'almost podium' riders (many 4th places)
    almost_podium = no_podium_riders[no_podium_riders['fourth_places'] >= 5].sort_values('fourth_places', ascending=False)
    print(f"\n🥉 'ALMOST PODIUM' ANALYSIS (5+ fourth places):")
    print(f"Riders with 5+ fourth places but no podiums: {len(almost_podium)}")
    
    if len(almost_podium) > 0:
        print(f"Top 'almost podium' riders:")
        for i, (_, rider) in enumerate(almost_podium.head(10).iterrows(), 1):
            name = rider['rider_clean']
            country = rider['country_clean']
            fourths = int(rider['fourth_places'])
            total_races = int(rider['total_races'])
            fourth_rate = (fourths / total_races) * 100 if total_races > 0 else 0
            print(f"  {i:2d}. {name} ({country}): {fourths} quartos lugares ({fourth_rate:.1f}% das corridas)")
    
    # Country analysis
    print(f"\n🌍 GEOGRAPHICAL ANALYSIS:")
    country_analysis = no_podium_riders.groupby('country_clean').agg({
        'rider_clean': 'count',
        'non_podium_points': 'sum'
    }).sort_values('non_podium_points', ascending=False)
    
    country_analysis.columns = ['riders_count', 'total_points_finishes']
    
    print(f"Top 10 países com mais posições 4º-6º (pilotos sem pódio):")
    for i, (country, data) in enumerate(country_analysis.head(10).iterrows(), 1):
        riders = data['riders_count']
        points = data['total_points_finishes']
        avg_per_rider = points / riders
        print(f"  {i:2d}. {country}: {points} posições ({riders} pilotos, {avg_per_rider:.1f} média/piloto)")
    
    # Visualization 1: Top riders
    plt.figure(figsize=(14, 8))
    top_10 = no_podium_sorted.head(10)
    plt.barh(range(len(top_10)), top_10['non_podium_points'], color='lightcoral')
    plt.yticks(range(len(top_10)), [name[:15] + '...' if len(name) > 15 else name for name in top_10['rider_clean']])
    plt.xlabel('Total de Posições 4º-6º')
    plt.title('Top 10 Pilotos com Mais Posições 4º-6º (Sem Pódios)')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
    
    # Visualization 2: Position breakdown for top 10
    fig, ax = plt.subplots(figsize=(14, 8))
    top_10 = no_podium_sorted.head(10)
    
    fourth_vals = top_10['fourth_places'].values
    fifth_vals = top_10['fifth_places'].values
    sixth_vals = top_10['sixth_places'].values
    
    x = range(len(top_10))
    ax.bar(x, fourth_vals, label='4º lugar', alpha=0.8, color='#FF6B6B')
    ax.bar(x, fifth_vals, bottom=fourth_vals, label='5º lugar', alpha=0.8, color='#4ECDC4')
    ax.bar(x, sixth_vals, bottom=fourth_vals + fifth_vals, label='6º lugar', alpha=0.8, color='#45B7D1')
    
    ax.set_xticks(x)
    ax.set_xticklabels([name[:10] + '...' if len(name) > 10 else name for name in top_10['rider_clean']], rotation=45)
    ax.set_ylabel('Número de Posições')
    ax.set_title('Distribuição de Posições 4º-6º por Piloto (Top 10 - Sem Pódios)')
    ax.legend()
    plt.tight_layout()
    plt.show()
    
    # Visualization 3: Distribution histogram
    plt.figure(figsize=(12, 6))
    plt.hist(no_podium_riders['non_podium_points'], bins=20, alpha=0.7, color='lightcoral', edgecolor='black')
    plt.axvline(no_podium_riders['non_podium_points'].mean(), color='red', linestyle='--', 
                label=f'Mean: {no_podium_riders["non_podium_points"].mean():.1f}')
    plt.axvline(no_podium_riders['non_podium_points'].median(), color='green', linestyle='--', 
                label=f'Median: {no_podium_riders["non_podium_points"].median():.1f}')
    plt.xlabel('Número de Posições 4º-6º')
    plt.ylabel('Número de Pilotos')
    plt.title('Distribuição de Posições 4º-6º entre Pilotos Sem Pódios')
    plt.legend()
    plt.tight_layout()
    plt.show()
    
else:
    print("\n❌ RESPOSTA Q18: Não foram encontrados pilotos sem pódios com dados de posições")

## 2. Q19: Pilotos com maior número de 2º e 3º que nunca venceram?

Comprehensive analysis of riders who achieved podium success but never won a race.

In [ ]:
print("=== Q19: PILOTOS COM MAIS 2º E 3º SEM VITÓRIAS ===")

# Filter riders who never won
no_wins_riders = df[df['victories'] == 0].copy()

if len(no_wins_riders) > 0:
    print(f"📊 STATISTICAL OVERVIEW:")
    print(f"Total riders analyzed: {len(df)}")
    print(f"Riders who never won a race: {len(no_wins_riders)} ({len(no_wins_riders)/len(df)*100:.1f}%)")
    
    # Calculate 2nd + 3rd places
    no_wins_riders['second_third_total'] = no_wins_riders['second_places'] + no_wins_riders['third_places']
    total_second_third = no_wins_riders['second_third_total'].sum()
    
    print(f"Total 2nd and 3rd finishes (no wins group): {total_second_third:,}")
    print(f"Average 2nd+3rd per rider: {no_wins_riders['second_third_total'].mean():.1f}")
    print(f"Riders with podiums but no wins: {(no_wins_riders['total_podiums'] > 0).sum()}")
    
    # Sort by total 2nd and 3rd places
    no_wins_sorted = no_wins_riders.sort_values('second_third_total', ascending=False)
    
    print(f"\nTop 20 pilotos com mais 2º e 3º lugares sem nunca vencer:")
    top_20 = no_wins_sorted.head(20)
    for i, (_, rider) in enumerate(top_20.iterrows(), 1):
        name = rider['rider_clean']
        country = rider['country_clean']
        second = int(rider['second_places'])
        third = int(rider['third_places'])
        total_23 = int(rider['second_third_total'])
        total_podiums = int(rider['total_podiums'])
        total_races = int(rider['total_races'])
        podium_rate = (total_podiums / total_races) * 100 if total_races > 0 else 0
        print(f"{i:2d}. {name} ({country}): {total_23} posições (2º:{second}, 3º:{third}) | {total_podiums} pódios em {total_races} corridas ({podium_rate:.1f}%)")
    
    # Champion analysis
    champion = no_wins_sorted.iloc[0]
    champion_name = champion['rider_clean']
    champion_country = champion['country_clean']
    champion_second = int(champion['second_places'])
    champion_third = int(champion['third_places'])
    champion_total_23 = int(champion['second_third_total'])
    champion_total_podiums = int(champion['total_podiums'])
    champion_total_races = int(champion['total_races'])
    
    print(f"\n🏆 RESPOSTA Q19:")
    print(f"Piloto com mais 2º e 3º lugares sem vitórias: {champion_name} ({champion_country})")
    print(f"Total de 2º e 3º lugares: {champion_total_23}")
    print(f"Detalhamento: 2º lugar = {champion_second}, 3º lugar = {champion_third}")
    print(f"Total de pódios: {champion_total_podiums}")
    print(f"Total de corridas: {champion_total_races}")
    
    # Advanced analysis of champion
    podium_rate = (champion_total_podiums / champion_total_races) * 100
    second_third_rate = (champion_total_23 / champion_total_races) * 100
    print(f"\nAnálise detalhada de {champion_name}:")
    print(f"  Taxa de pódios: {podium_rate:.1f}%")
    print(f"  Taxa de 2º/3º lugares: {second_third_rate:.1f}%")
    print(f"  Eficiência pódio: Nunca converteu em vitória")
    
    # Position preference analysis for champion
    if champion_second > champion_third:
        preferred_position = "2º lugar"
        preference_ratio = champion_second / champion_third if champion_third > 0 else float('inf')
    elif champion_third > champion_second:
        preferred_position = "3º lugar"
        preference_ratio = champion_third / champion_second if champion_second > 0 else float('inf')
    else:
        preferred_position = "Equilibrado"
        preference_ratio = 1.0
    
    print(f"  Posição preferencial: {preferred_position}")
    if preference_ratio != float('inf') and preference_ratio != 1.0:
        print(f"  Ratio de preferência: {preference_ratio:.2f}x")
    
    # Statistical analysis of the no-wins group
    print(f"\n📈 STATISTICAL ANALYSIS - NO WINS GROUP:")
    
    # Distribution stats
    print(f"Distribution of 2nd+3rd finishes:")
    print(f"  Mean: {no_wins_riders['second_third_total'].mean():.1f}")
    print(f"  Median: {no_wins_riders['second_third_total'].median():.1f}")
    print(f"  Std Dev: {no_wins_riders['second_third_total'].std():.1f}")
    print(f"  Max: {no_wins_riders['second_third_total'].max()}")
    
    # Percentile analysis
    percentiles = [50, 75, 90, 95, 99]
    print(f"\nPercentiles (2nd+3rd finishes):")
    for p in percentiles:
        value = np.percentile(no_wins_riders['second_third_total'], p)
        print(f"  {p}th percentile: {value:.0f} finishes")
    
    # "Eternal runner-up" analysis
    print(f"\n🥈 'ETERNAL RUNNER-UP' ANALYSIS:")
    almost_winners = no_wins_riders[no_wins_riders['second_places'] >= 5].sort_values('second_places', ascending=False)
    print(f"Riders with 5+ second places but no wins: {len(almost_winners)}")
    
    if len(almost_winners) > 0:
        print(f"\nTop 10 'eternal runners-up':")
        for i, (_, rider) in enumerate(almost_winners.head(10).iterrows(), 1):
            name = rider['rider_clean']
            country = rider['country_clean']
            seconds = int(rider['second_places'])
            thirds = int(rider['third_places'])
            total_races = int(rider['total_races'])
            second_rate = (seconds / total_races) * 100 if total_races > 0 else 0
            print(f"  {i:2d}. {name} ({country}): {seconds} segundos lugares, {thirds} terceiros ({second_rate:.1f}% 2º)")
    
    # Position distribution analysis
    print(f"\n🎯 POSITION DISTRIBUTION ANALYSIS:")
    total_seconds = no_wins_riders['second_places'].sum()
    total_thirds = no_wins_riders['third_places'].sum()
    
    print(f"Overall position distribution (no-wins group):")
    print(f"  2nd place: {total_seconds:,} ({total_seconds/total_second_third*100:.1f}%)")
    print(f"  3rd place: {total_thirds:,} ({total_thirds/total_second_third*100:.1f}%)")
    
    # Performance efficiency analysis
    print(f"\n⚡ PODIUM EFFICIENCY ANALYSIS:")
    podium_riders = no_wins_riders[no_wins_riders['total_podiums'] > 0]
    
    if len(podium_riders) > 0:
        print(f"Riders with podiums but no wins: {len(podium_riders)}")
        
        # Calculate podium efficiency metrics
        podium_riders = podium_riders.copy()
        podium_riders['podium_rate'] = podium_riders['total_podiums'] / podium_riders['total_races']
        podium_riders['second_rate'] = podium_riders['second_places'] / podium_riders['total_races']
        
        print(f"\nMost consistent podium finishers (no wins):")
        consistent_podium = podium_riders.nlargest(5, 'podium_rate')
        for i, (_, rider) in enumerate(consistent_podium.iterrows(), 1):
            name = rider['rider_clean']
            podium_rate = rider['podium_rate'] * 100
            total_podiums = int(rider['total_podiums'])
            total_races = int(rider['total_races'])
            print(f"  {i}. {name}: {podium_rate:.1f}% podium rate ({total_podiums}/{total_races})")
    
    # Country analysis
    print(f"\n🌍 GEOGRAPHICAL ANALYSIS:")
    country_analysis = no_wins_riders.groupby('country_clean').agg({
        'rider_clean': 'count',
        'second_third_total': 'sum',
        'total_podiums': 'sum'
    }).sort_values('second_third_total', ascending=False)
    
    country_analysis.columns = ['riders_count', 'total_2nd_3rd', 'total_podiums']
    
    print(f"Top 10 países com mais 2º e 3º lugares (pilotos sem vitória):")
    for i, (country, data) in enumerate(country_analysis.head(10).iterrows(), 1):
        riders = data['riders_count']
        positions = data['total_2nd_3rd']
        podiums = data['total_podiums']
        avg_per_rider = positions / riders
        print(f"  {i:2d}. {country}: {positions} posições ({riders} pilotos, {avg_per_rider:.1f} média, {podiums} pódios totais)")
    
    # Visualization 1: Top riders
    plt.figure(figsize=(14, 8))
    top_10 = no_wins_sorted.head(10)
    plt.barh(range(len(top_10)), top_10['second_third_total'], color='lightblue')
    plt.yticks(range(len(top_10)), [name[:15] + '...' if len(name) > 15 else name for name in top_10['rider_clean']])
    plt.xlabel('Total de 2º e 3º Lugares')
    plt.title('Top 10 Pilotos com Mais 2º e 3º Lugares (Sem Vitórias)')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
    
    # Visualization 2: 2nd vs 3rd breakdown
    fig, ax = plt.subplots(figsize=(14, 8))
    top_10 = no_wins_sorted.head(10)
    
    second_vals = top_10['second_places'].values
    third_vals = top_10['third_places'].values
    
    x = range(len(top_10))
    ax.bar(x, second_vals, label='2º lugar', alpha=0.8, color='silver')
    ax.bar(x, third_vals, bottom=second_vals, label='3º lugar', alpha=0.8, color='#CD7F32')
    
    ax.set_xticks(x)
    ax.set_xticklabels([name[:10] + '...' if len(name) > 10 else name for name in top_10['rider_clean']], rotation=45)
    ax.set_ylabel('Número de Posições')
    ax.set_title('Distribuição de 2º e 3º Lugares por Piloto (Top 10 - Sem Vitórias)')
    ax.legend()
    plt.tight_layout()
    plt.show()
    
    # Visualization 3: Runner-up analysis
    if len(almost_winners) >= 5:
        plt.figure(figsize=(12, 6))
        top_5_runners_up = almost_winners.head(5)
        plt.bar(range(len(top_5_runners_up)), top_5_runners_up['second_places'], color='silver', alpha=0.7)
        plt.xticks(range(len(top_5_runners_up)), 
                  [name[:10] + '...' if len(name) > 10 else name for name in top_5_runners_up['rider_clean']], 
                  rotation=45)
        plt.ylabel('Número de 2º Lugares')
        plt.title('Top 5 "Eternos Vice-Campeões" - Mais 2º Lugares Sem Vitórias')
        plt.tight_layout()
        plt.show()
    
else:
    print("\n❌ RESPOSTA Q19: Não foram encontrados pilotos sem vitórias com dados de posições")

## 3. Performance Tier Analysis

Advanced modeling to identify different performance tiers based on finishing positions.

In [ ]:
print("=== PERFORMANCE TIER ANALYSIS ===")

# Create comprehensive performance tiers
print(f"\n🏆 COMPREHENSIVE PERFORMANCE TIER MODELING:")

def assign_performance_tier(rider):
    """Assign performance tier based on finishing positions"""
    victories = rider['victories']
    podiums = rider['total_podiums']
    points_finishes = rider['points_finishes']
    total_races = rider['total_races']
    
    if victories >= 20:
        return 'Tier 1: Elite Winners'
    elif victories >= 5:
        return 'Tier 2: Regular Winners'
    elif victories >= 1:
        return 'Tier 3: Race Winners'
    elif podiums >= 10:
        return 'Tier 4: Podium Regulars'
    elif podiums >= 1:
        return 'Tier 5: Podium Finishers'
    elif points_finishes >= 20:
        return 'Tier 6: Consistent Scorers'
    elif points_finishes >= 5:
        return 'Tier 7: Points Scorers'
    else:
        return 'Tier 8: Limited Results'

# Apply tier assignment
df_tiers = df.copy()
df_tiers['performance_tier'] = df_tiers.apply(assign_performance_tier, axis=1)

# Tier analysis
tier_analysis = df_tiers.groupby('performance_tier').agg({
    'rider_clean': 'count',
    'victories': ['sum', 'mean'],
    'total_podiums': ['sum', 'mean'],
    'points_finishes': ['sum', 'mean'],
    'total_races': ['sum', 'mean']
})

# Flatten column names
tier_analysis.columns = ['rider_count', 'total_victories', 'avg_victories', 
                        'total_podiums', 'avg_podiums', 'total_points_finishes', 
                        'avg_points_finishes', 'total_races', 'avg_races']

# Sort by tier level
tier_order = ['Tier 1: Elite Winners', 'Tier 2: Regular Winners', 'Tier 3: Race Winners',
              'Tier 4: Podium Regulars', 'Tier 5: Podium Finishers', 'Tier 6: Consistent Scorers',
              'Tier 7: Points Scorers', 'Tier 8: Limited Results']

tier_analysis = tier_analysis.reindex(tier_order)

print(f"Performance tier distribution:")
total_riders = len(df_tiers)
for tier, data in tier_analysis.iterrows():
    count = int(data['rider_count'])
    percentage = (count / total_riders) * 100
    avg_victories = data['avg_victories']
    avg_podiums = data['avg_podiums']
    print(f"{tier}: {count} riders ({percentage:.1f}%) - Avg: {avg_victories:.1f}V, {avg_podiums:.1f}P")

# Success rate analysis
print(f"\n📊 SUCCESS RATE ANALYSIS:")
winners = tier_analysis.loc[tier_analysis.index.str.contains('Winners')]['rider_count'].sum()
podium_finishers = tier_analysis.loc[tier_analysis.index.str.contains('Podium')]['rider_count'].sum()
scorers = tier_analysis.loc[tier_analysis.index.str.contains('Scorers')]['rider_count'].sum()

print(f"Race winner rate: {winners/total_riders*100:.1f}% ({winners} riders)")
print(f"Podium finisher rate: {podium_finishers/total_riders*100:.1f}% ({podium_finishers} riders)")
print(f"Points scorer rate: {scorers/total_riders*100:.1f}% ({scorers} riders)")
print(f"Success rate (winners + podium finishers): {(winners+podium_finishers)/total_riders*100:.1f}%")

# Tier transition analysis - riders close to next tier
print(f"\n🎯 TIER TRANSITION ANALYSIS:")
print(f"Riders close to promotion to next tier:")

# Tier 7 → Tier 6 (5-19 points finishes, need 20+)
tier_7_riders = df_tiers[df_tiers['performance_tier'] == 'Tier 7: Points Scorers']
close_to_tier_6 = tier_7_riders[tier_7_riders['points_finishes'] >= 15]
print(f"  Tier 7→6: {len(close_to_tier_6)} riders with 15+ points (need 20)")

# Tier 5 → Tier 4 (1-9 podiums, need 10+)
tier_5_riders = df_tiers[df_tiers['performance_tier'] == 'Tier 5: Podium Finishers']
close_to_tier_4 = tier_5_riders[tier_5_riders['total_podiums'] >= 7]
print(f"  Tier 5→4: {len(close_to_tier_4)} riders with 7+ podiums (need 10)")

# Tier 3 → Tier 2 (1-4 victories, need 5+)
tier_3_riders = df_tiers[df_tiers['performance_tier'] == 'Tier 3: Race Winners']
close_to_tier_2 = tier_3_riders[tier_3_riders['victories'] >= 3]
print(f"  Tier 3→2: {len(close_to_tier_2)} riders with 3+ victories (need 5)")

# Show examples for each transition
if len(close_to_tier_6) > 0:
    print(f"\n  Examples close to Tier 6:")
    for _, rider in close_to_tier_6.head(3).iterrows():
        print(f"    {rider['rider_clean']}: {int(rider['points_finishes'])} points finishes")

if len(close_to_tier_4) > 0:
    print(f"\n  Examples close to Tier 4:")
    for _, rider in close_to_tier_4.head(3).iterrows():
        print(f"    {rider['rider_clean']}: {int(rider['total_podiums'])} podiums")

if len(close_to_tier_2) > 0:
    print(f"\n  Examples close to Tier 2:")
    for _, rider in close_to_tier_2.head(3).iterrows():
        print(f"    {rider['rider_clean']}: {int(rider['victories'])} victories")

# Visualization: Tier distribution
plt.figure(figsize=(14, 8))
tier_counts = df_tiers['performance_tier'].value_counts()[tier_order]
plt.bar(range(len(tier_counts)), tier_counts.values, color='skyblue', alpha=0.7)
plt.xticks(range(len(tier_counts)), [tier.replace('Tier ', 'T').replace(': ', '\n') for tier in tier_counts.index], 
           rotation=45)
plt.ylabel('Number of Riders')
plt.title('Distribution of Riders Across Performance Tiers')
plt.tight_layout()
plt.show()

## 4. Never-Won-Never-Podium Analysis

Special analysis of riders who achieved consistent results without major success.

In [ ]:
print("=== NEVER-WON-NEVER-PODIUM ANALYSIS ===")

# Identify different categories of underperformers relative to participation
print(f"\n🔍 DETAILED UNDERPERFORMANCE ANALYSIS:")

# Category 1: High participation, no podiums
high_participation_no_podium = df[
    (df['total_races'] >= 50) & 
    (df['total_podiums'] == 0)
].sort_values('total_races', ascending=False)

print(f"High Participation, No Podiums (50+ races):")
print(f"  Count: {len(high_participation_no_podium)} riders")
if len(high_participation_no_podium) > 0:
    print(f"  Examples:")
    for i, (_, rider) in enumerate(high_participation_no_podium.head(5).iterrows(), 1):
        name = rider['rider_clean']
        races = int(rider['total_races'])
        points = int(rider['points_finishes'])
        points_rate = (points / races) * 100
        print(f"    {i}. {name}: {races} races, {points} points finishes ({points_rate:.1f}%)")

# Category 2: Moderate participation, consistent points
consistent_points_no_podium = df[
    (df['total_races'] >= 20) & 
    (df['total_races'] < 50) &
    (df['points_finishes'] >= 10) &
    (df['total_podiums'] == 0)
].sort_values('points_finishes', ascending=False)

print(f"\nConsistent Points, No Podiums (20-49 races, 10+ points):")
print(f"  Count: {len(consistent_points_no_podium)} riders")
if len(consistent_points_no_podium) > 0:
    print(f"  Examples:")
    for i, (_, rider) in enumerate(consistent_points_no_podium.head(5).iterrows(), 1):
        name = rider['rider_clean']
        races = int(rider['total_races'])
        points = int(rider['points_finishes'])
        best_finish = "6th" if rider['sixth_places'] > 0 else "5th" if rider['fifth_places'] > 0 else "4th" if rider['fourth_places'] > 0 else "Unknown"
        print(f"    {i}. {name}: {races} races, {points} points, best: {best_finish}")

# Category 3: "Almost there" riders (many 4th places)
almost_podium_specialists = df[
    (df['fourth_places'] >= 5) & 
    (df['total_podiums'] == 0)
].sort_values('fourth_places', ascending=False)

print(f"\n'Almost Podium' Specialists (5+ fourth places, no podiums):")
print(f"  Count: {len(almost_podium_specialists)} riders")
if len(almost_podium_specialists) > 0:
    print(f"  Top specialists:")
    for i, (_, rider) in enumerate(almost_podium_specialists.head(5).iterrows(), 1):
        name = rider['rider_clean']
        fourths = int(rider['fourth_places'])
        races = int(rider['total_races'])
        fourth_rate = (fourths / races) * 100
        print(f"    {i}. {name}: {fourths} fourth places in {races} races ({fourth_rate:.1f}%)")

# Performance efficiency analysis
print(f"\n📊 EFFICIENCY ANALYSIS FOR NON-PODIUM RIDERS:")

non_podium_riders = df[df['total_podiums'] == 0].copy()
non_podium_riders['efficiency_score'] = (
    non_podium_riders['fourth_places'] * 4 +
    non_podium_riders['fifth_places'] * 3 +
    non_podium_riders['sixth_places'] * 2
) / non_podium_riders['total_races']

# Replace infinite values with 0
non_podium_riders['efficiency_score'] = non_podium_riders['efficiency_score'].replace([np.inf, -np.inf], 0)

most_efficient_no_podium = non_podium_riders.nlargest(10, 'efficiency_score')

print(f"Most efficient non-podium riders (weighted points per race):")
for i, (_, rider) in enumerate(most_efficient_no_podium.iterrows(), 1):
    name = rider['rider_clean']
    efficiency = rider['efficiency_score']
    races = int(rider['total_races'])
    points = int(rider['points_finishes'])
    if races > 0 and efficiency > 0:
        print(f"  {i:2d}. {name}: {efficiency:.2f} efficiency ({points}/{races} points rate)")

# Regional analysis of non-achievers
print(f"\n🌍 REGIONAL ANALYSIS - NON-PODIUM RIDERS:")
regional_analysis = non_podium_riders.groupby('country_clean').agg({
    'rider_clean': 'count',
    'points_finishes': 'sum',
    'total_races': 'sum'
}).sort_values('points_finishes', ascending=False)

regional_analysis.columns = ['rider_count', 'total_points', 'total_races']
regional_analysis['avg_points_per_rider'] = regional_analysis['total_points'] / regional_analysis['rider_count']
regional_analysis['overall_points_rate'] = regional_analysis['total_points'] / regional_analysis['total_races'] * 100

print(f"Top 10 countries by total points (non-podium riders):")
for i, (country, data) in enumerate(regional_analysis.head(10).iterrows(), 1):
    riders = int(data['rider_count'])
    points = int(data['total_points'])
    avg_points = data['avg_points_per_rider']
    points_rate = data['overall_points_rate']
    print(f"  {i:2d}. {country}: {points} points ({riders} riders, {avg_points:.1f} avg, {points_rate:.1f}% rate)")

# Visualization: Non-podium rider efficiency
if len(most_efficient_no_podium) > 0:
    plt.figure(figsize=(12, 8))
    valid_efficient = most_efficient_no_podium[most_efficient_no_podium['efficiency_score'] > 0]
    if len(valid_efficient) > 0:
        plt.barh(range(len(valid_efficient)), valid_efficient['efficiency_score'], color='lightgreen', alpha=0.7)
        plt.yticks(range(len(valid_efficient)), 
                  [name[:15] + '...' if len(name) > 15 else name for name in valid_efficient['rider_clean']])
        plt.xlabel('Efficiency Score (Weighted Points per Race)')
        plt.title('Most Efficient Non-Podium Riders')
        plt.gca().invert_yaxis()
        plt.tight_layout()
        plt.show()

# Comparative analysis: podium vs non-podium groups
print(f"\n📈 COMPARATIVE ANALYSIS: PODIUM vs NON-PODIUM:")
podium_riders = df[df['total_podiums'] > 0]

comparison_stats = {
    'Group': ['Podium Riders', 'Non-Podium Riders'],
    'Count': [len(podium_riders), len(non_podium_riders)],
    'Avg_Races': [podium_riders['total_races'].mean(), non_podium_riders['total_races'].mean()],
    'Avg_Points': [podium_riders['points_finishes'].mean(), non_podium_riders['points_finishes'].mean()],
    'Points_Rate': [podium_riders['points_rate'].mean() * 100, non_podium_riders['points_rate'].mean() * 100]
}

comparison_df = pd.DataFrame(comparison_stats)
print(comparison_df.round(1).to_string(index=False))

print(f"\nKey differences:")
race_diff = comparison_df.loc[0, 'Avg_Races'] - comparison_df.loc[1, 'Avg_Races']
points_diff = comparison_df.loc[0, 'Avg_Points'] - comparison_df.loc[1, 'Avg_Points']
rate_diff = comparison_df.loc[0, 'Points_Rate'] - comparison_df.loc[1, 'Points_Rate']

print(f"  - Podium riders race {race_diff:.1f} more races on average")
print(f"  - Podium riders score {points_diff:.1f} more points finishes on average")
print(f"  - Podium riders have {rate_diff:.1f}% higher points scoring rate")

## 5. Model Summary and Key Findings

Consolidation of all finishing positions modeling results and insights.

In [ ]:
print("=" * 80)
print("FINISHING POSITIONS MODELING - SUMMARY OF FINDINGS")
print("=" * 80)

print("\n🏆 BUSINESS QUESTIONS ANSWERED:")

print("\nQ18 - Piloto com mais 4º-6º sem pódios:")
if 'no_podium_sorted' in locals() and len(no_podium_sorted) > 0:
    q18_champion = no_podium_sorted.iloc[0]
    print(f"     → {q18_champion['rider_clean']} ({q18_champion['country_clean']}) com {int(q18_champion['non_podium_points'])} posições 4º-6º")
    print(f"       Detalhes: {int(q18_champion['fourth_places'])} quartos, {int(q18_champion['fifth_places'])} quintos, {int(q18_champion['sixth_places'])} sextos")
    print(f"       Taxa de pontuação: {q18_champion['non_podium_points']/q18_champion['total_races']*100:.1f}% em {int(q18_champion['total_races'])} corridas")
else:
    print("     → Dados não disponíveis")

print("\nQ19 - Piloto com mais 2º-3º sem vitórias:")
if 'no_wins_sorted' in locals() and len(no_wins_sorted) > 0:
    q19_champion = no_wins_sorted.iloc[0]
    print(f"     → {q19_champion['rider_clean']} ({q19_champion['country_clean']}) com {int(q19_champion['second_third_total'])} posições 2º-3º")
    print(f"       Detalhes: {int(q19_champion['second_places'])} segundos lugares, {int(q19_champion['third_places'])} terceiros lugares")
    print(f"       Taxa de pódio: {q19_champion['total_podiums']/q19_champion['total_races']*100:.1f}% em {int(q19_champion['total_races'])} corridas")
else:
    print("     → Dados não disponíveis")

print("\n📊 KEY STATISTICAL INSIGHTS:")

if 'no_podium_riders' in locals():
    print(f"\nNon-Podium Riders Analysis:")
    print(f"  - Total riders without podiums: {len(no_podium_riders):,} ({len(no_podium_riders)/len(df)*100:.1f}% of all riders)")
    print(f"  - Total 4th-6th finishes: {no_podium_riders['non_podium_points'].sum():,}")
    print(f"  - Average 4th-6th per rider: {no_podium_riders['non_podium_points'].mean():.1f}")
    print(f"  - Most 4th-6th finishes: {no_podium_riders['non_podium_points'].max()} (single rider)")

if 'no_wins_riders' in locals():
    print(f"\nNon-Winner Riders Analysis:")
    print(f"  - Total riders without wins: {len(no_wins_riders):,} ({len(no_wins_riders)/len(df)*100:.1f}% of all riders)")
    print(f"  - Riders with podiums but no wins: {(no_wins_riders['total_podiums'] > 0).sum():,}")
    print(f"  - Total 2nd-3rd finishes: {no_wins_riders['second_third_total'].sum():,}")
    print(f"  - Most 2nd-3rd finishes: {no_wins_riders['second_third_total'].max()} (single rider)")

if 'tier_analysis' in locals():
    print(f"\nPerformance Tier Distribution:")
    elite_tiers = tier_analysis.loc[tier_analysis.index.str.contains('Elite|Regular|Race')]['rider_count'].sum()
    podium_tiers = tier_analysis.loc[tier_analysis.index.str.contains('Podium')]['rider_count'].sum()
    points_tiers = tier_analysis.loc[tier_analysis.index.str.contains('Scorers|Consistent')]['rider_count'].sum()
    
    print(f"  - Elite performers (winners): {elite_tiers} riders ({elite_tiers/len(df)*100:.1f}%)")
    print(f"  - Podium finishers: {podium_tiers} riders ({podium_tiers/len(df)*100:.1f}%)")
    print(f"  - Points scorers only: {points_tiers} riders ({points_tiers/len(df)*100:.1f}%)")
    print(f"  - Success rate (elite + podium): {(elite_tiers+podium_tiers)/len(df)*100:.1f}%")

print("\n🔍 SPECIALIZED INSIGHTS:")

if 'almost_podium' in locals() and len(almost_podium) > 0:
    print(f"\n'Almost Podium' Specialists:")
    print(f"  - Riders with 5+ fourth places (no podiums): {len(almost_podium)}")
    if len(almost_podium) > 0:
        top_almost = almost_podium.iloc[0]
        print(f"  - Leading specialist: {top_almost['rider_clean']} with {int(top_almost['fourth_places'])} fourth places")

if 'almost_winners' in locals() and len(almost_winners) > 0:
    print(f"\n'Eternal Runners-Up':")
    print(f"  - Riders with 5+ second places (no wins): {len(almost_winners)}")
    if len(almost_winners) > 0:
        top_runner_up = almost_winners.iloc[0]
        print(f"  - Leading runner-up: {top_runner_up['rider_clean']} with {int(top_runner_up['second_places'])} second places")

if 'most_efficient_no_podium' in locals() and len(most_efficient_no_podium) > 0:
    print(f"\nEfficiency Analysis:")
    top_efficient = most_efficient_no_podium.iloc[0]
    print(f"  - Most efficient non-podium rider: {top_efficient['rider_clean']}")
    print(f"  - Efficiency score: {top_efficient['efficiency_score']:.3f} (weighted points per race)")

print("\n📈 MODELING QUALITY ASSESSMENT:")
print(f"  - Dataset completeness: {100 - (df.isnull().sum().sum() / df.size * 100):.1f}%")
print(f"  - Sample size: {len(df):,} riders")
print(f"  - Position coverage: Complete (1st through 6th places)")
print(f"  - Statistical power: HIGH (comprehensive position data)")

print("\n⚠️  LIMITATIONS AND ASSUMPTIONS:")
print("  - Data limited to top 6 finishing positions")
print("  - Points system assumptions (6th place = points scoring)")
print("  - No temporal context for career progression")
print("  - Aggregated data may mask seasonal variations")

print("\n💡 BUSINESS IMPLICATIONS:")
print("  1. Majority of riders never achieve podium success")
print(f"  2. Small percentage ({elite_tiers/len(df)*100:.1f}%) achieve elite status")
print("  3. 'Almost' categories represent significant talent pools")
print("  4. Consistent points scoring without podiums is common")
print("  5. Clear performance hierarchies exist in motorcycle racing")

print("\n🚀 RECOMMENDATIONS FOR INTEGRATION:")
print("  1. Cross-reference with riders_info for career context")
print("  2. Combine with race_winners for complete success picture")
print("  3. Use tier modeling for talent development strategies")
print("  4. Analyze 'almost' categories for breakthrough potential")

print("\n✅ FINISHING POSITIONS MODELING COMPLETED")
print("This dataset provides comprehensive insights into rider performance distribution and achievement patterns.")